# 🎬 Reels — Free-GPU Talking Avatar (MuseTalk) — Kaggle Edition

Generate a talking-avatar reel on a **free Kaggle GPU (T4/P100)**.

**Accelerator → GPU T4 x2** before running, then **Run All** — zero manual steps needed.

Pipeline: `script → cloned voice (XTTS) → MuseTalk lip-sync → 9:16 reel → Instagram post`.

Video + voice are downloaded automatically from the URLs set in the dashboard Source section.
No dataset upload required.

In [ ]:
# 1) Check GPU
!nvidia-smi -L
import sys; print('python', sys.version.split()[0])

In [ ]:
# 2) Get repo + MuseTalk
import os
WORK = '/kaggle/working'
if not os.path.isdir(f'{WORK}/reels'):
    !git clone https://github.com/amsinghnavdeep/reels.git {WORK}/reels
%cd {WORK}/reels
!git pull -q
!pip -q install edge-tts pydub pyyaml
os.makedirs('engines', exist_ok=True)
if not os.path.isdir('engines/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git engines/MuseTalk

In [ ]:
%%bash
# 3) Isolated Python-3.10 env for MuseTalk (~8-10 min, one-time)
set -e
WORK=/kaggle/working
MF=/opt/conda
if [ ! -x $MF/bin/conda ]; then
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/mf.sh
  bash /tmp/mf.sh -b -p $MF
fi
source $MF/etc/profile.d/conda.sh
conda create -y -n muse python=3.10 >/dev/null 2>&1 || true
conda activate muse
cd $WORK/reels/engines/MuseTalk
grep -viE '^(torch|torchvision|torchaudio|mmcv|mmdet|mmpose|mmengine)' requirements.txt > /tmp/req.txt || cp requirements.txt /tmp/req.txt
pip -q install -r /tmp/req.txt
pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
pip -q install -U openmim
mim install mmengine
mim install 'mmcv==2.1.0'
mim install 'mmdet==3.1.0'
pip -q install "setuptools<70" wheel
pip -q install --no-build-isolation chumpy
mim install 'mmpose==1.1.0'
sed -i "s/mmcv_maximum_version = '2.1.0'/mmcv_maximum_version = '2.2.0'/" /opt/conda/envs/muse/lib/python3.10/site-packages/mmdet/__init__.py
export MPLBACKEND=Agg
python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"
python -c "from mmpose.apis import inference_topdown; print('mmpose import OK')"

In [ ]:
# 4) Download MuseTalk weights (~5 GB, one-time)
WORK = '/kaggle/working'
%cd {WORK}/reels/engines/MuseTalk
import os
from huggingface_hub import hf_hub_download
os.makedirs('models/face-parse-bisent', exist_ok=True)
def dl(repo, filename, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    p = hf_hub_download(repo_id=repo, filename=filename, local_dir=local_dir)
    print('  ok:', p)
for f in ['musetalk/musetalk.json', 'musetalk/pytorch_model.bin',
          'musetalkV15/musetalk.json', 'musetalkV15/unet.pth']:
    dl('TMElyralab/MuseTalk', f, 'models')
for f in ['config.json', 'diffusion_pytorch_model.bin']:
    dl('stabilityai/sd-vae-ft-mse', f, 'models/sd-vae')
for f in ['config.json', 'pytorch_model.bin', 'preprocessor_config.json']:
    dl('openai/whisper-tiny', f, 'models/whisper')
dl('yzd-v/DWPose', 'dw-ll_ucoco_384.pth', 'models/dwpose')
!gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
!curl -sL https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth
import glob
print('\nweights present:')
for p in sorted(glob.glob('models/**/*', recursive=True)):
    if os.path.isfile(p): print(' ', p, round(os.path.getsize(p)/1e6,1), 'MB')

In [ ]:
# 5) SOURCE VIDEO + VOICE — downloaded automatically from the dashboard config.
#    NO manual upload or Kaggle dataset needed. Run All works with zero human touch.
#    To change files: update URLs in dashboard → Step 1 → Source → Save config.
WORK = '/kaggle/working'
%cd {WORK}/reels
import os, subprocess, json, urllib.request
os.makedirs('output', exist_ok=True)

API_URL    = os.environ.get('API',        'https://reels-api.navdeepcingh1.workers.dev')
RUNNER_KEY = os.environ.get('RUNNER_KEY', 'f601e7ea2613add02dd244fcaa28be1aedbe306cf3076427')

# ── Pull config from dashboard ───────────────────────────────────────────────
req = urllib.request.Request(f'{API_URL}/job', headers={'x-runner-key': RUNNER_KEY})
try:
    with urllib.request.urlopen(req, timeout=15) as r:
        cfg = json.loads(r.read())
    VIDEO_URL  = cfg.get('video_url', '')
    VOICE_URL  = cfg.get('voice_url', '')
    SCRIPT_TXT = cfg.get('script', '')
    TRIM       = cfg.get('trim', '')
    CROP       = cfg.get('crop', '')
    LANG       = cfg.get('lang', 'hi')
    print(f'✅ Config loaded from dashboard')
    print(f'   video_url : {VIDEO_URL}')
    print(f'   voice_url : {VOICE_URL}')
    print(f'   script    : {SCRIPT_TXT[:80]}...')
except Exception as e:
    print(f'⚠️  Could not reach dashboard ({e}) — using fallback hardcoded URLs')
    VIDEO_URL  = 'https://drive.google.com/uc?export=download&id=1WSEY4HFJjNnGjZ3wPzxsZXiP0N7gTfgz'
    VOICE_URL  = 'https://drive.google.com/uc?export=download&id=1ox0KMYSrkOlVX-hoJoBxOgf71AMmlmEV'
    SCRIPT_TXT = open('examples/script.txt').read() if os.path.exists('examples/script.txt') else ''
    TRIM, CROP, LANG = '', '', 'hi'

# ── Write script to file ─────────────────────────────────────────────────────
if SCRIPT_TXT:
    os.makedirs('examples', exist_ok=True)
    open('examples/script.txt', 'w').write(SCRIPT_TXT)
    print(f'   script.txt written ({len(SCRIPT_TXT)} chars)')
SCRIPT = 'examples/script.txt'

# ── Download + normalise video ───────────────────────────────────────────────
assert VIDEO_URL, 'No video URL. Set one in the dashboard → Step 1 → Source.'
subprocess.run(f'wget -q --show-progress -O output/drive_raw.mp4 "{VIDEO_URL}"', shell=True, check=True)
_dn = 'scale=1280:1280:force_original_aspect_ratio=decrease:force_divisible_by=2'
vf  = f'-vf "{CROP},{_dn}"' if CROP else f'-vf "{_dn}"'
subprocess.run(
    f'ffmpeg -y {TRIM} -i output/drive_raw.mp4 {vf} -an -r 25 -c:v libx264 -pix_fmt yuv420p output/drive_clip.mp4',
    shell=True, check=True)
DRIVE_VIDEO = 'output/drive_clip.mp4'
print('✅ Video ready:', DRIVE_VIDEO)

# ── Download + normalise voice ───────────────────────────────────────────────
assert VOICE_URL, 'No voice URL. Set one in the dashboard → Step 1 → Source.'
subprocess.run(f'wget -q --show-progress -O output/voice_raw.m4a "{VOICE_URL}"', shell=True, check=True)
subprocess.run('ffmpeg -y -i output/voice_raw.m4a -ar 16000 -ac 1 output/voice_ref.wav', shell=True, check=True)
print('✅ Voice ready: output/voice_ref.wav')

# ── Clone voice with XTTS-v2 ─────────────────────────────────────────────────
!source /opt/conda/etc/profile.d/conda.sh && \
  (conda env list | grep -q '/envs/tts$' || conda create -y -q -n tts python=3.10) && \
  conda activate tts && \
  export MPLBACKEND=Agg && \
  pip -q install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 && \
  pip -q install transformers==4.35.2 && \
  pip -q install coqui-tts==0.22.1 && \
  COQUI_TOS_AGREED=1 python /kaggle/working/reels/tts.py \
    --script /kaggle/working/reels/examples/script.txt \
    --engine xtts --clone /kaggle/working/reels/output/voice_ref.wav --lang hi \
    --output /kaggle/working/reels/output/speech_raw.wav
import os; assert os.path.exists('output/speech_raw.wav'), 'TTS failed — see logs above'

# ── Strip silence ─────────────────────────────────────────────────────────────
subprocess.run('ffmpeg -y -i output/speech_raw.wav -af '
  '"silenceremove=start_periods=1:start_silence=0.05:start_threshold=-45dB:'
  'stop_periods=-1:stop_silence=0.2:stop_threshold=-45dB" '
  '-ar 16000 -ac 1 output/speech.wav', shell=True, check=True)
print('✅ Speech ready: output/speech.wav')

In [ ]:
# 6) MuseTalk lip-sync
WORK = '/kaggle/working'
import yaml, os
os.chdir(f'{WORK}/reels/engines/MuseTalk')
os.environ['FFMPEG_PATH'] = '/usr/bin'
os.environ['MPLBACKEND']  = 'Agg'
cfg = {'task_0': {'video_path': f'{WORK}/reels/{DRIVE_VIDEO}', 'audio_path': f'{WORK}/reels/output/speech.wav'}}
os.makedirs('configs/inference', exist_ok=True)
open('configs/inference/reels.yaml', 'w').write(yaml.dump(cfg))
!sed -i 's|^            shutil.rmtree(save_dir_full)|            if get_file_type(video_path) == "video": shutil.rmtree(save_dir_full)|' scripts/inference.py
!MPLBACKEND=Agg /opt/conda/envs/muse/bin/python -m scripts.inference \
  --inference_config configs/inference/reels.yaml \
  --result_dir /kaggle/working/reels/output/muse \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 --fps 25 --batch_size 4 --use_float16 \
  --parsing_mode jaw --extra_margin 10

In [ ]:
# 6b) SHARPEN — GFPGAN face restore
WORK = '/kaggle/working'
import os, glob
os.chdir(f'{WORK}/reels/engines/MuseTalk')
cands = sorted(glob.glob(f'{WORK}/reels/output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
assert cands, 'No MuseTalk output — run cell 6 first.'
RAW = cands[-1]; print('raw MuseTalk clip:', RAW)

!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  pip -q install gfpgan realesrgan basicsr facexlib && \
  BS=$(/opt/conda/envs/muse/bin/python -c "import basicsr,os;print(os.path.dirname(basicsr.__file__))") && \
  sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' \
    $BS/data/degradations.py 2>/dev/null; echo patched

os.makedirs(f'{WORK}/reels/output/enh_in',  exist_ok=True)
os.makedirs(f'{WORK}/reels/output/enh_out', exist_ok=True)
!rm -f {WORK}/reels/output/enh_in/* {WORK}/reels/output/enh_out/*
!ffmpeg -y -i "$RAW" -qscale:v 1 {WORK}/reels/output/enh_in/f%06d.png
open(f'{WORK}/reels/output/_restore.py', 'w').write(r'''
import os, glob, cv2
from gfpgan import GFPGANer
root = "/kaggle/working/reels"
gfp = GFPGANer(model_path="https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
               upscale=2, arch="clean", channel_multiplier=2, bg_upsampler=None)
frames = sorted(glob.glob(root + "/output/enh_in/*.png"))
for i, fp in enumerate(frames):
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    _, _, out = gfp.enhance(img, has_aligned=False, only_center_face=True, paste_back=True)
    cv2.imwrite(root + "/output/enh_out/" + os.path.basename(fp), out)
    if i % 50 == 0: print("  restored", i, "/", len(frames))
print("done", len(frames), "frames")
''')
!source /opt/conda/etc/profile.d/conda.sh && conda activate muse && \
  MPLBACKEND=Agg python {WORK}/reels/output/_restore.py
!ffmpeg -y -framerate 25 -i {WORK}/reels/output/enh_out/f%06d.png \
  -i {WORK}/reels/output/speech.wav \
  -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest {WORK}/reels/output/muse_hd.mp4
print('✅ Enhanced clip:', f'{WORK}/reels/output/muse_hd.mp4')

In [ ]:
# 7) Compose 9:16 reel
WORK = '/kaggle/working'
%cd {WORK}/reels
import glob, os
hd = 'output/muse_hd.mp4'
if os.path.exists(hd):
    raw = hd
else:
    clips = sorted(glob.glob('output/muse/**/*.mp4', recursive=True), key=os.path.getmtime)
    assert clips, 'No MuseTalk output — check cell 6 logs.'
    raw = clips[-1]
print('composing from:', raw)
!python reel_utils.py --in "$raw" --out output/reel.mp4
print('✅ Reel ready: output/reel.mp4')
print('Download it from: Kaggle → Output panel → output/reel.mp4')

In [ ]:
# 8) DASHBOARD ASSETS
#    Video + voice are fetched automatically from the public URLs set in the
#    dashboard Source section. No Kaggle dataset upload needed.
#    To swap files: update URLs in the dashboard → Save config → queue a new job.
WORK = '/kaggle/working'
import os
os.makedirs(f'{WORK}/reels/reels_state', exist_ok=True)
print('✅ Asset folder ready. URLs pulled from dashboard — no dataset upload required.')

In [ ]:
# 9) DASHBOARD RUNNER
#    Connects this Kaggle GPU to your Cloudflare dashboard.
#    ONESHOT=1 → runs one job then exits cleanly (ideal for Kaggle scheduled notebooks).
#
#    Full automated flow:
#      Dashboard queues job  →  runner picks it up  →  downloads video + voice
#      →  runs full pipeline  →  posts to Instagram  →  reports done ✅
import os
os.environ['API']        = 'https://reels-api.navdeepcingh1.workers.dev'
os.environ['RUNNER_KEY'] = 'f601e7ea2613add02dd244fcaa28be1aedbe306cf3076427'
os.environ['ONESHOT']    = '1'  # Kaggle: run one job then exit (saves GPU quota)
WORK = '/kaggle/working'
!cd {WORK}/reels && pip -q install instagrapi >/dev/null 2>&1; python runner.py